Historical study notebook. Its original input is unavailable in this public edition, so this is not a runnable public demo. It may require an input with the original structure and dependencies. This work used supplied learning resources and starter material; retained attribution appears with the notebook content.

# ============================================
# COMPARING OPEN-SOURCE EMBEDDING MODELS FOR RAG
# ============================================

This notebook compares three popular embedding models to see which performs best for pharmaceutical document retrieval.

You'll learn:
- How different embedding models produce different results
- How to measure retrieval time and quality
- How to choose the right model for your use case
# ============================================

In [ ]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing packages for RAG and embedding models:
# - llama-index: Core RAG framework
# - llama-index-embeddings-huggingface: Access to HuggingFace models
# - pymupdf: PDF text extraction
# - nest_asyncio: Handles async event loops in Colab
#
# WHY THIS MATTERS:
# These libraries let us test multiple embedding models easily.
#
# WHAT YOU'LL SEE:
# Installation progress (the -q flag keeps output minimal)
# ============================================

!pip install -q llama-index llama-index-embeddings-huggingface pymupdf
!pip install -q nest_asyncio

In [ ]:
# ============================================
# STEP 2: Import Libraries and Configure Settings
# ============================================
#
# WHAT WE'RE DOING:
# Loading all required modules and configuring the environment.
#
# WHY THIS MATTERS:
# - nest_asyncio.apply() fixes event loop issues in Colab/Jupyter
# - Settings.llm = None disables the LLM (we're only testing retrieval,
#   not generation, so we don't need an LLM for this notebook)
#
# WHAT YOU'LL SEE:
# A message about "MockLLM" - this is expected and harmless.
# It just means LlamaIndex is using a placeholder since we disabled the LLM.
# ============================================

import nest_asyncio
nest_asyncio.apply()

from llama_index.core import VectorStoreIndex, Document, Settings, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
import fitz  # PyMuPDF
import time

# Disable LLM - we're only comparing embedding models for retrieval
Settings.llm = None

# ============================================
# SECTION 2: Load the Pharmaceutical Document
# ============================================
#
# We'll extract text from a pharmaceutical SDF document
# to use as our test corpus for embedding comparison.
# ============================================

In [ ]:
# ============================================
# STEP 3: Load and Extract Text from PDF
# ============================================
#
# WHAT WE'RE DOING:
# Loading the pharmaceutical SDF document and extracting all text.
#
# WHY THIS MATTERS:
# This document contains product specifications, test methods,
# and quality control information - realistic content for
# testing pharmaceutical document retrieval.
#
# WHAT YOU'LL SEE:
# Word count from the extracted document.
# ============================================

# Load the pharmaceutical SDF document
pdf_path = "your-document.pdf"
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the pharmaceutical document.")

In [ ]:
# ============================================
# STEP 4: Split Document into Chunks
# ============================================
#
# WHAT WE'RE DOING:
# Breaking the document into overlapping chunks.
# We'll build the index separately for each embedding model.
#
# WHY THIS MATTERS:
# Chunking affects retrieval quality. Small chunks (50 tokens)
# with overlap (50 tokens) give precise results while maintaining context.
#
# WHAT YOU'LL SEE:
# Confirmation of chunk creation.
# ============================================

# Define a sentence splitter
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.")

In [ ]:
!pip install -q llama-index-llms-gemini



In [ ]:
import os
from google.colab import userdata
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

Settings.llm = Gemini(model="models/gemini-flash-latest")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Rebuild the index now that embed model + LLM are set
index = VectorStoreIndex(nodes)
print("Index ready with MiniLM embeddings + Gemini LLM.")


In [ ]:
query = "What test methods were used for quality control?"
retriever = index.as_retriever(similarity_top_k=8)
for node in retriever.retrieve(query):
    print(f"Score: {node.score:.3f}")


In [ ]:
import time
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank

query = "What test methods were used for quality control?"

configs = [
    {"name": "A (Default)", "top_k": 5, "threshold": None, "rerank": False},
    {"name": "B",           "top_k": 8, "threshold": 0.25, "rerank": False},
    {"name": "C",           "top_k": 5, "threshold": 0.25, "rerank": True},
]

def query_with_retry(engine, query, tries=5):
    for attempt in range(tries):
        try:
            return engine.query(query)
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 60
                print(f"  Rate limited — waiting {wait}s (attempt {attempt+1}/{tries})...")
                time.sleep(wait)
            else:
                raise
    return "ERROR: still rate-limited after retries"

for cfg in configs:
    print("\n" + "="*70)
    print(f"CONFIG {cfg['name']} | top_k={cfg['top_k']} threshold={cfg['threshold']} rerank={cfg['rerank']}")
    print("="*70)

    retriever = index.as_retriever(similarity_top_k=cfg["top_k"])
    nodes_out = retriever.retrieve(query)

    if cfg["threshold"] is not None:
        nodes_out = [n for n in nodes_out if n.score and n.score > cfg["threshold"]]

    postprocessors = []
    if cfg["rerank"]:
        postprocessors.append(
            SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=3)
        )

    print(f"\nChunks retrieved: {len(nodes_out)}")
    for i, n in enumerate(nodes_out):
        print(f"  Chunk {i+1} (Score: {n.score:.2f}): {n.get_text()[:150]}...")

    engine = RetrieverQueryEngine.from_args(retriever=retriever, node_postprocessors=postprocessors)
    response = query_with_retry(engine, query)
    print(f"\nFINAL ANSWER:\n{response}\n")

    time.sleep(20)   # pause between configs to respect the free-tier limit
